# Period data

MODFLOW 6 stress packages take their period data in one of two forms:

* **lists** (CHD, WEL, DRN, RIV, GHB, RCH, EVT, ...): a period block has a row
  per boundary feature, naming its cell and giving its values;
* **arrays** (CHDG, WELG, RCHA, EVTA, ...): a period block has a value per
  cell of the grid.

### Imports

In [1]:
import numpy as np

In [2]:
from flopy4.mf6.constants import FILL_DNODATA
from flopy4.mf6.gwf import Evt, Gwf, Wel
from flopy4.mf6.simulation import Simulation
from flopy4.mf6.utils.grid import StructuredGrid
from flopy4.mf6.utils.time import Time

### A small model

One layer of 3 rows and 4 columns, run for 3 stress periods. Packages
attached to the model get their dimensions from it.

In [3]:
grid = StructuredGrid(
    nlay=1,
    nrow=3,
    ncol=4,
    delr=np.ones(4),
    delc=np.ones(3),
    top=np.ones((3, 4)),
    botm=np.zeros((1, 3, 4)),
)
sim = Simulation(name="period-data", tdis=Time(perlen=[1.0] * 3, nstp=[1] * 3))
gwf = Gwf(parent=sim, name="model", dis=grid)

### Rows

Period data maps each (0-based) period to its rows. A row is a tuple of the
package's columns, in MF6 input guide order: for WEL, the cell, `q`, the
auxiliary values and the boundname. A string in place of a number names a
time series.

Periods fill forward: a period without a block repeats the last one, and an
empty list turns every feature off. Here period 1 fills forward from period
0, and period 2 has two wells in one cell.

In [4]:
wel = Wel(
    parent=gwf,
    auxiliary=["conc"],
    boundnames=True,
    stress_period_data={
        0: [
            ((0, 0, 0), -1.0, (5.0,), "north"),
            ((0, 2, 3), "pumping", (0.0,), None),
        ],
        2: [
            ((0, 1, 1), -2.0, (1.0,), "east"),
            ((0, 1, 1), -0.5, (2.0,), "east-2"),
        ],
    },
)

Rows come back as `Wel.StressPeriodData` objects, which can also be built
by name.

In [5]:
row = wel.stress_period_data[0][0]
print(row)
assert row.q == -1.0 and row.aux == (5.0,) and row.boundname == "north"
assert row == Wel.StressPeriodData(cellid=(0, 0, 0), q=-1.0, aux=(5.0,), boundname="north")
assert str(wel.stress_period_data[0][1].q) == "pumping"
assert list(wel.stress_period_data) == [0, 2]

Wel.StressPeriodData(cellid=(0, 0, 0), q=-1.0, aux=(5.0,), boundname='north')


### The array view

`to_xarray()` gives a variable per value column on `(per, layer, row, col)`,
with every period filled forward. Empty cells hold `FILL_DNODATA` (3e30).

* Each auxiliary variable is its own variable (`conc`).
* Boundnames are a `boundname` coordinate.
* A time series value is NaN, with its name in a `<var>_ts` coordinate.

Two rows in one cell raise by default; here they're summed.

In [6]:
ds = wel.to_xarray(duplicates="sum")
ds

<xarray.Dataset> Size: 1kB
Dimensions:    (per: 3, layer: 1, row: 3, col: 4)
Coordinates:
    q_ts       (per, layer, row, col) object 288B nan nan nan ... nan nan nan
    boundname  (per, layer, row, col) object 288B 'north' nan nan ... nan nan
Dimensions without coordinates: per, layer, row, col
Data variables:
    q          (per, layer, row, col) float64 288B -1.0 3e+30 ... 3e+30 3e+30
    conc       (per, layer, row, col) float64 288B 5.0 3e+30 ... 3e+30 3e+30

In [7]:
q = ds["q"]
assert q.dims == ("per", "layer", "row", "col")
assert q[1, 0, 0, 0] == -1.0  # period 1 filled forward from period 0
assert q[0, 0, 1, 1] == FILL_DNODATA  # no well here until period 2
assert q[2, 0, 1, 1] == -2.5  # two wells, summed
assert ds["conc"][0, 0, 0, 0] == 5.0
assert ds["boundname"][0, 0, 0, 0] == "north"

assert np.isnan(q[0, 0, 2, 3])  # a time series value...
assert ds["q_ts"][0, 0, 2, 3] == "pumping"  # ...and its name

Use `ds["head"]`, not `ds.head`, which is xarray's `Dataset.head()`.

### Editing and writing back

Edit the arrays, then pass the dataset to `from_xarray()`.

In [8]:
edited = ds.copy(deep=True)
edited["q"][1, 0, 0, 0] = -2.0
wel.from_xarray(edited)

`from_xarray()` writes a block only where a period changes, so period 1
now has its own block, and period 2's wells come back as one summed well.

In [9]:
for kper, rows in wel.stress_period_data.items():
    print(kper, rows)
assert list(wel.stress_period_data) == [0, 1, 2]
assert wel.stress_period_data[1][0].q == -2.0
assert len(wel.stress_period_data[2]) == 1

0 [Wel.StressPeriodData(cellid=(0, 0, 0), q=-1.0, aux=(5.0,), boundname='north'), Wel.StressPeriodData(cellid=(0, 2, 3), q=TimeSeriesRef(name='pumping'), aux=(0.0,), boundname=None)]
1 [Wel.StressPeriodData(cellid=(0, 0, 0), q=-2.0, aux=(5.0,), boundname='north'), Wel.StressPeriodData(cellid=(0, 2, 3), q=TimeSeriesRef(name='pumping'), aux=(0.0,), boundname=None)]
2 [Wel.StressPeriodData(cellid=(0, 1, 1), q=-2.5, aux=(3.0,), boundname='east-2')]


Summing adds auxiliary values too, and keeps the last row's boundname.

In [10]:
assert wel.stress_period_data[2][0].aux == (3.0,)
assert wel.stress_period_data[2][0].boundname == "east-2"

To add a feature, set all of its variables at the cell. Setting only some
raises an error naming the missing one.

In [11]:
edited = wel.to_xarray()
edited["q"][0, 0, 2, 0] = -4.0
try:
    wel.from_xarray(edited)
except ValueError as e:
    print(e)
edited["conc"][0, 0, 2, 0] = 1.0
wel.from_xarray(edited)
assert len(wel.stress_period_data[0]) == 3

aux has no value at (0, 2, 0) in period 0, where other columns do


### Two features in one cell

* `duplicates="sum"`, `"first"` or `"last"` merges them. `"sum"` can't sum
  a time series. The boundname kept is the first row's for `"first"`, the
  last row's otherwise.
* `sparse=True` keeps every row, on an `entry` dim with the period and cell
  as coordinates. This also suits packages without a single cell per row
  (HFB, MVR, OC).

In [12]:
wel = Wel(
    parent=gwf,
    stress_period_data={0: [((0, 1, 1), -2.0), ((0, 1, 1), -0.5)]},
)
try:
    wel.to_xarray()
except ValueError as e:
    print(e)
assert wel.to_xarray(duplicates="last")["q"][0, 0, 1, 1] == -0.5

period 0 sets q at (0, 1, 1) twice; to_xarray(sparse=True) keeps every row, or duplicates='sum', 'first' or 'last' merges them


In [13]:
sparse = wel.to_xarray(sparse=True)
sparse

<xarray.Dataset> Size: 80B
Dimensions:       (entry: 2)
Coordinates:
    per           (entry) int64 16B 0 0
    cellid_layer  (entry) int64 16B 0 0
    cellid_row    (entry) int64 16B 1 1
    cellid_col    (entry) int64 16B 1 1
Dimensions without coordinates: entry
Data variables:
    q             (entry) float64 16B -2.0 -0.5
Attributes:
    periods:  [0]

In [14]:
assert sparse["q"].dims == ("entry",)
assert list(sparse["q"].values) == [-2.0, -0.5]
assert list(sparse["cellid_row"].values) == [1, 1]

expected = wel.stress_period_data
wel.from_xarray(sparse)
assert wel.stress_period_data == expected

### Several values in one column

With `nseg` segments, each EVT row gives `nseg - 1` depth (`pxdp`) and rate
(`petm`) proportions as tuples.

In [15]:
evt = Evt(
    parent=gwf,
    nseg=3,
    stress_period_data={
        0: [
            # cell, surface, rate, depth, pxdp, petm
            ((0, 0, 0), 1.0, 0.1, 2.0, (0.3, 0.6), (0.5, 0.2)),
            ((0, 0, 1), 1.0, 0.1, 2.0, (0.4, 0.7), (0.6, 0.3)),
        ]
    },
)
assert evt.stress_period_data[0][0].pxdp == (0.3, 0.6)

In the array view these get an extra `nseg-1` dim. A cell needs all of its
segment values.

In [16]:
ds = evt.to_xarray()
pxdp = ds["pxdp"]
assert pxdp.dims == ("per", "nseg-1", "layer", "row", "col")
pxdp.isel(per=0, layer=0)

<xarray.DataArray 'pxdp' (nseg-1: 2, row: 3, col: 4)> Size: 192B
array([[[3.e-01, 4.e-01, 3.e+30, 3.e+30],
        [3.e+30, 3.e+30, 3.e+30, 3.e+30],
        [3.e+30, 3.e+30, 3.e+30, 3.e+30]],

       [[6.e-01, 7.e-01, 3.e+30, 3.e+30],
        [3.e+30, 3.e+30, 3.e+30, 3.e+30],
        [3.e+30, 3.e+30, 3.e+30, 3.e+30]]])
Dimensions without coordinates: nseg-1, row, col

In [17]:
np.testing.assert_array_equal(pxdp[0, :, 0, 0, 0], [0.3, 0.6])

expected = evt.stress_period_data
evt.from_xarray(ds)
assert evt.stress_period_data == expected

### Lists or arrays?

The rows remain the package's data; the array view is for working with it.
For a stress over most of the grid, like recharge, an array package (RCHA,
EVTA, CHDG, WELG, ...) is smaller and faster to write. For features in a
few cells, use a list package. Only list packages support several features
per cell, boundnames and EVT segments.